In [1]:
using DelimitedFiles
using Statistics
using Printf
using CairoMakie

**Problem 4**

2D heterogenous related need the pre-coded from HW5 write up inside `funcs/flow_transport_2d.jl`

In [2]:
include("funcs/flow_transport_2d.jl")

track_particles (generic function with 1 method)

With the given parameters

In [4]:
# physical properties
ρ = 1000 # water density [kg/m^3]
μ = 1.0e-3 # dynamic viscosity [Pa s]
g = 9.81 # gravitational acceleration [m/s^2]
ϕ = 0.3 # porosity [-]

# model domain
Lx = 100 # domain length in x [m]
Ly = 50 # domain length in y [m]
nx = 100 # number of cells in x 
ny = 50 # number of cells in y
dx = Lx / nx # = 1 m
dy = Ly / ny # = 1 m

# boundary conditions
hL = 10 # left boundary head [m] (Dirichlet)
hR = 9 # right boundary head [m] (Dirichlet)
c_in = 0.0 # inflowing water
day = 24 * 60 * 60 # [s]

86400

porous medium given in the file permeability_field.csv

In [9]:
## get permeability fields
kmat = readdlm("data/permeability_field.csv", ',', Float64) # [m^2]
μ_log10k = -10 # mean of log10(k)
σ_log10k = 1 # std log10(k)
# field
Y = (log10.(kmat) .- μ_log10k) ./ σ_log10k

# heatmap(Y) # just to preview

50×100 Matrix{Float64}:
  0.306827  -0.239832    -0.829539  …  -0.23351     -0.0258339  -0.064376
 -0.581875  -0.447926    -0.481728     -0.507339    -0.4361     -0.382959
 -0.900708  -1.10387     -0.503877     -0.675224    -0.649555   -0.963423
 -1.61304   -1.32149     -1.58576      -1.1078      -1.02521    -0.989753
 -1.85529   -1.95647     -2.24739      -0.761562    -0.673463   -0.633617
 -1.47169   -2.04894     -2.11588   …  -0.00246762  -0.0633912   0.0753059
 -2.60413   -2.6293      -2.72365       0.169317     0.495711    0.668013
 -1.74672   -2.15435     -2.10418       0.0415761    0.288417    0.582694
 -1.52883   -1.48813     -1.63954      -0.17271     -0.137576    0.28235
 -1.33349   -1.71988     -1.91494       0.466869     0.163347   -0.345263
  ⋮                                 ⋱                           
  1.70726    1.30855      1.40478      -0.899262    -0.783751   -0.627033
  0.909275   0.664009     0.382267     -0.466568    -0.762336   -1.0843
  0.242317   0.00685738  

Change the heterogeneity of the permeability fields ($\sigma_{\log_{10} k} = 0.1,\ 0.5,\ 1,\ 2$) using the formula

$$\log_{10} k = \mu_{\log_{10} k} + \sigma_{\log_{10} k}\, Y$$

In [10]:
σ_ls = [0.1, 0.5, 1, 2] # heterogeneity levels
log10k(σ) = μ_log10k .+ σ .* Y

log10k (generic function with 1 method)

Setting up the coordinate system for plotting

In [11]:
X = range(dx/2, step=dx, length=nx) # cell-center coordinates [m]
Ycoord = range(dy/2, step=dy, length=ny)

0.5:1.0:49.5

Solve the steady-state flow equation 1st as in `HW4` and `HW5`, and retain the face-centered Darcy velocities for the transport calculation. Then, solve the advection equation (i.e., the ADE without dispersion flux) using a first-order upwind scheme for the advective fluxes and an explicit time-stepping scheme (forward Euler). The initial concentration field is:

$$
c(x, y, t = 0) =
\begin{cases}
1, & 0 \le x \le 10\ \text{m} \\
0, & 10 < x \le 100\ \text{m}
\end{cases}
$$

At the left boundary, the inflowing water has $c=0$ for all $t>0$. Use the Darcy velocities at cell faces (u) to calculate the advective solute mass fluxes.

In [12]:
## initial plume: c = 1 for 0 <= x <= 10 m
C0 = zeros(ny, nx)
C0[:, X .<= 10] .= 1
M0 = ϕ * sum(C0) * dx * dy # initial solute mass (per unit thickness)

150.0

In parallel, solve the advective particle tracking, where the solute particles move along with pore velocities. Distribute uniformly 20,000 particles over the upstream domain $0 \le x \le 10\ \text{m}$, $0 \le y \le 50\ \text{m}$ at time $t = 0$.

In [ ]:
# 20,000 particles on a uniform 100 x 200 lattice over 0-10 m x 0-50 m
# (0.1 m x 0.25 m spacing, i.e. 40 particles per cell, 2,000 per 1-m column)
xp_ls = range(0.05, step = 0.1, length = 100)
yp_ls = range(0.125, step = 0.25, length = 200)
xp0 = vec([x for y in yp_ls, x in xp_ls])
yp0 = vec([y for y in yp_ls, x in xp_ls])
np = length(xp0)
np_per_m = np / 10 # = 2,000, normalizes the particle counts per 1-m interval
